# Missouri Voter Resource Allocation — Geo Pipeline

Builds precinct-level feature datasets for 2016, 2020, and 2024 Missouri elections.

**Pipeline stages:**
- Stage 0 — Configuration & Imports
- Stage 1 — Load VEST Precinct Shapefiles
- Stage 2 — Load Census Block Data
- Stage 3 — Inspect VEST Column Structure
- Stage 4 — Create Precinct Identifiers (composite_prec_id)
- Stage 5 — Load ACS Staging Data
- Stage 6 — Build Full Precinct Features
- Stage 7 — County Aggregation
- Stage 8 — Save Outputs

## Stage 0 — Configuration & Imports

In [1]:
import sys
import re
import logging
import pandas as pd
import geopandas as gpd
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# Assumes notebook lives in <project_root>/notebooks/
PROJECT_ROOT = Path('..').resolve()
DATA_ROOT    = PROJECT_ROOT / 'data'
GEO_RAW      = DATA_ROOT / 'geo' / 'raw'
PROCESSED    = DATA_ROOT / 'processed'
GEO_OUT      = DATA_ROOT / 'geo' / 'output'
GEO_OUT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from geo_loader          import GeoLoader
from census_block_loader import CensusBlockLoader
from precinct_builder    import (
    build_precinct_features,
    aggregate_to_county,
    COUNTY_FIPS_CANDIDATES,
)

# Get a free Census API key at: https://api.census.gov/data/key_signup.html
# Only needed on first run; after that the data is cached locally.
CENSUS_API_KEY = 'YOUR_CENSUS_API_KEY_HERE'

ELECTION_YEARS = [2016, 2020, 2024]

print('Configuration complete.')
print(f'  Project root : {PROJECT_ROOT}')
print(f'  GEO raw dir  : {GEO_RAW}')
print(f'  Processed dir: {PROCESSED}')
print(f'  GEO output   : {GEO_OUT}')

Configuration complete.
  Project root : C:\Users\daves\OneDrive\Documents\School\SP26\AnalyticsApps\final\AnalyticsApps
  GEO raw dir  : C:\Users\daves\OneDrive\Documents\School\SP26\AnalyticsApps\final\AnalyticsApps\data\geo\raw
  Processed dir: C:\Users\daves\OneDrive\Documents\School\SP26\AnalyticsApps\final\AnalyticsApps\data\processed
  GEO output   : C:\Users\daves\OneDrive\Documents\School\SP26\AnalyticsApps\final\AnalyticsApps\data\geo\output


## Stage 1 — Load VEST Precinct Shapefiles

In [2]:
geo_loader = GeoLoader(geo_raw_dir=str(GEO_RAW))
vest = {}

for year in ELECTION_YEARS:
    print(f'\nLoading {year} VEST shapefile...')
    vest[year] = geo_loader.get_precinct_shapefile(year)
    print(f'  {year}: {len(vest[year]):,} rows | CRS = {vest[year].crs}')
    print(f'  Columns: {vest[year].columns.tolist()}')

print('\nAll VEST shapefiles loaded.')


Loading 2016 VEST shapefile...
    -> Loading shapefile: mo_vest_16.shp
  2016: 3,324 rows | CRS = EPSG:4269
  Columns: ['STATEFP', 'COUNTYFP', 'NAME', 'G16PRERTRU', 'G16PREDCLI', 'G16PRELJOH', 'G16PREGSTE', 'G16PRECCAS', 'G16USSRBLU', 'G16USSDKAN', 'G16USSLDIN', 'G16USSGMCF', 'G16USSCRYM', 'G16GOVRGRE', 'G16GOVDKOS', 'G16GOVLSPR', 'G16GOVGFIT', 'G16GOVITUR', 'G16LTGRPAR', 'G16LTGDCAR', 'G16LTGLHED', 'G16LTGGLEA', 'G16ATGRHAW', 'G16ATGDHEN', 'G16TRERSCH', 'G16TREDBAK', 'G16TRELOTO', 'G16TREGHEX', 'G16SOSRASH', 'G16SOSDSMI', 'G16SOSLMOR', 'geometry']

Loading 2020 VEST shapefile...
    -> Loading shapefile: mo_vest_20.shp
  2020: 3,733 rows | CRS = EPSG:4269
  Columns: ['STATEFP', 'COUNTYFP', 'NAME', 'G20PRERTRU', 'G20PREDBID', 'G20PRELJOR', 'G20PREGHAW', 'G20PRECBLA', 'G20GOVRPAR', 'G20GOVDGAL', 'G20GOVLCOM', 'G20GOVGBAU', 'G20LTGRKEH', 'G20LTGDCAN', 'G20LTGLSLA', 'G20LTGGDRA', 'G20ATGRSCH', 'G20ATGDFIN', 'G20ATGLBAB', 'G20SOSRASH', 'G20SOSDFAL', 'G20SOSLFRE', 'G20SOSGLEH', 'G20SOSCVE

## Stage 2 — Load Census Block Data

Downloads Missouri 2020 Census block geometries (~60 MB) and block-level population
from the Census PL 94-171 API on first run. Both are cached locally after that.

In [3]:
print('Loading census block data...')
block_loader = CensusBlockLoader(
    geo_raw_dir=str(GEO_RAW),
    census_api_key=CENSUS_API_KEY,
)
blocks_gdf = block_loader.get_blocks_with_population()

print(f'\nBlocks loaded : {len(blocks_gdf):,} rows')
print(f'Columns       : {blocks_gdf.columns.tolist()}')
print(f'CRS           : {blocks_gdf.crs}')
print(f'Total pop     : {blocks_gdf["total_population"].sum():,.0f}')
print(f'Total VAP     : {blocks_gdf["vap_total"].sum():,.0f}')

INFO: Loading cached block geometry from disk...


Loading census block data...


INFO: Loading cached block population data from disk...
INFO: Census blocks loaded: 253,632 blocks, total MO population: 6,154,913



Blocks loaded : 253,632 rows
Columns       : ['GEOID20', 'geometry', 'total_population', 'vap_total', 'state', 'county', 'tract', 'block']
CRS           : EPSG:4269
Total pop     : 6,154,913
Total VAP     : 4,775,612


## Stage 3 — Inspect VEST Column Structure

Confirms presidential vote columns exist for each year and identifies the
county FIPS column (name varies across VEST vintages).

In [4]:
print('=== Presidential vote column check ===')

for year in ELECTION_YEARS:
    suffix   = str(year)[2:]
    pre_cols = [c for c in vest[year].columns if re.match(rf'^G{suffix}PRE[A-Z]+$', c)]

    county_col = next(
        (c for c in COUNTY_FIPS_CANDIDATES if c in vest[year].columns), 'NOT FOUND'
    )

    text_cols = [
        c for c in vest[year].select_dtypes(include='object').columns
        if c != 'geometry'
    ]

    print(f'\n{year}:')
    print(f'  County column          : {county_col}')
    print(f'  Presidential cols ({len(pre_cols):2d}) : {pre_cols}')
    print(f'  Text columns           : {text_cols}')

=== Presidential vote column check ===

2016:
  County column          : COUNTYFP
  Presidential cols ( 5) : ['G16PRERTRU', 'G16PREDCLI', 'G16PRELJOH', 'G16PREGSTE', 'G16PRECCAS']
  Text columns           : ['STATEFP', 'COUNTYFP', 'NAME']

2020:
  County column          : COUNTYFP
  Presidential cols ( 5) : ['G20PRERTRU', 'G20PREDBID', 'G20PRELJOR', 'G20PREGHAW', 'G20PRECBLA']
  Text columns           : ['STATEFP', 'COUNTYFP', 'NAME']

2024:
  County column          : COUNTYFP
  Presidential cols ( 9) : ['G24PREDHAR', 'G24PREGSTE', 'G24PRELOLI', 'G24PREOAYY', 'G24PREOCRU', 'G24PREOPOT', 'G24PREOSON', 'G24PREOWRI', 'G24PRERTRU']
  Text columns           : ['UNIQUE_ID', 'COUNTYFP', 'County', 'Elec_Muni', 'Prec_Code', 'Precinct']


## Stage 4 — Create Precinct Identifiers

Each VEST vintage uses a different column for the precinct label:
- 2016 and 2020 use `NAME`; 2024 uses `Precinct`

We build `composite_prec_id` = `COUNTYFP_<precinct_name>` — a consistent,
year-independent identifier for every precinct.

**Split precincts** (non-contiguous territory stored as separate rows sharing
the same name) are dissolved: geometries unioned, vote columns summed.
2016 has 1 split pair; 2024 has ~257; 2020 is already clean.

In [5]:
# Step 1: Create composite_prec_id with year-appropriate precinct name column
PRECINCT_NAME_COL = {
    2016: 'NAME',
    2020: 'NAME',
    2024: 'Precinct',
}

for year in ELECTION_YEARS:
    vest[year] = vest[year].copy()
    vest[year]['composite_prec_id'] = (
        vest[year]['COUNTYFP'].str.zfill(3)
        + '_'
        + vest[year][PRECINCT_NAME_COL[year]].astype(str)
    )
    print(f'{year}: composite_prec_id created ({len(vest[year]):,} rows before dissolve)')


# Step 2: Dissolve split precincts
def dissolve_split_precincts(vest_gdf, precinct_id_col):
    """Union geometry and sum vote totals for split (multi-polygon) precincts."""
    vote_cols  = [c for c in vest_gdf.columns if re.match(r'^G\d{2}', c)]
    other_cols = [
        c for c in vest_gdf.columns
        if c not in vote_cols + ['geometry', precinct_id_col]
    ]
    agg = {col: 'sum'   for col in vote_cols}
    agg.update({col: 'first' for col in other_cols})
    return vest_gdf.dissolve(by=precinct_id_col, aggfunc=agg).reset_index()


for year in [2016, 2024]:   # 2020 is already clean
    n_before = len(vest[year])
    vest[year] = dissolve_split_precincts(vest[year], 'composite_prec_id')
    n_after    = len(vest[year])
    print(f'{year}: {n_before:,} -> {n_after:,} rows after dissolving split precincts')


# Step 3: Set the working precinct ID column for all downstream stages
PRECINCT_ID_COL = 'composite_prec_id'


# Step 4: Verify uniqueness
print('\n=== Precinct ID uniqueness check ===')
for year in ELECTION_YEARS:
    n = len(vest[year])
    u = vest[year][PRECINCT_ID_COL].nunique()
    status = 'OK' if n == u else f'WARNING: {n - u} duplicates remain!'
    print(f'  {year}: {n:,} rows, {u:,} unique  [{status}]')

2016: composite_prec_id created (3,324 rows before dissolve)
2020: composite_prec_id created (3,733 rows before dissolve)
2024: composite_prec_id created (3,333 rows before dissolve)
2016: 3,324 -> 3,323 rows after dissolving split precincts
2024: 3,333 -> 3,076 rows after dissolving split precincts

=== Precinct ID uniqueness check ===
  2016: 3,323 rows, 3,323 unique  [OK]
  2020: 3,733 rows, 3,733 unique  [OK]
  2024: 3,076 rows, 3,076 unique  [OK]


## Stage 5 — Load ACS Staging Data

Loads county-level ACS staging CSVs from `data/processed/`.
These are downscaled to precinct level via population-weighted apportionment
inside `build_precinct_features`.

In [6]:
acs_files = {
    'income'    : 'stg_census_income.csv',
    'education' : 'stg_census_education.csv',
    'race'      : 'stg_census_race.csv',
    'commute'   : 'stg_census_commute.csv',
    'sex_age'   : 'stg_census_sex_age.csv',
}

acs_staging_dfs = {}

for category, filename in acs_files.items():
    path = PROCESSED / filename
    df   = pd.read_csv(path, dtype={'county_fips': str})
    df   = df[df['census_year'].isin(ELECTION_YEARS)].copy()
    acs_staging_dfs[category] = df
    print(f'  {category:10s}: {len(df):,} rows | columns: {df.columns.tolist()}')

print(f'\n{len(acs_staging_dfs)} ACS staging datasets loaded.')

  income    : 345 rows | columns: ['census_year', 'county_fips', 'county_name', 'county_clean', 'median_household_income']
  education : 345 rows | columns: ['census_year', 'county_fips', 'county_name', 'county_clean', 'total_pop_25_plus', 'pct_bachelors_plus']
  race      : 345 rows | columns: ['census_year', 'county_fips', 'county_name', 'county_clean', 'total_population', 'pct_white', 'pct_minority']
  commute   : 345 rows | columns: ['census_year', 'county_fips', 'county_name', 'county_clean', 'total_workers', 'pct_no_vehicle']
  sex_age   : 345 rows | columns: ['census_year', 'county_fips', 'county_name', 'county_clean', 'total_population', 'voting_age_population', 'pct_voting_age']

5 ACS staging datasets loaded.


## Stage 6 — Build Full Precinct Features

`build_precinct_features` orchestrates CRS alignment, area-weighted block
apportionment, vote extraction, turnout calculation, and ACS downscaling
for each year in sequence.

The `gpd.overlay` intersection in step 2 may take **1–3 minutes per year**.

In [7]:
precinct_features = {}

for year in ELECTION_YEARS:
    precinct_features[year] = build_precinct_features(
        vest_gdf        = vest[year],
        blocks_gdf      = blocks_gdf,
        acs_staging_dfs = acs_staging_dfs,
        year            = year,
        precinct_id_col = PRECINCT_ID_COL,
    )

print('\n=== Precinct feature summary ===')
for year in ELECTION_YEARS:
    gdf = precinct_features[year]
    print(f'  {year}: {len(gdf):,} precincts | {len(gdf.columns)} columns')
    if 'turnout_pct' in gdf.columns:
        med = gdf['turnout_pct'].dropna().median()
        print(f'    Median turnout: {med:.1f}%')
    print(f'    Columns: {gdf.columns.tolist()}')


Building precinct features for 2016...
  [1/4] Aligning coordinate reference systems...


INFO: Computing block → precinct apportionment via area intersection...


  [2/4] Apportioning census blocks to precincts...


INFO:   Running gpd.overlay intersection (may take 1-2 minutes)...
INFO:   Apportionment complete. Total apportioned population: 6,154,910
INFO:   Found 5 presidential candidate columns for 2016.


  [3/4] Extracting vote totals and calculating turnout...
  [4/4] Downscaling ACS demographics to precinct level...
  Done. 3,323 precincts assembled for 2016.

Building precinct features for 2020...
  [1/4] Aligning coordinate reference systems...


...

  [3/4] Extracting vote totals and calculating turnout...
  [4/4] Downscaling ACS demographics to precinct level...
  Done. 3,733 precincts assembled for 2020.

Building precinct features for 2024...
  [1/4] Aligning coordinate reference systems...


INFO: Computing block → precinct apportionment via area intersection...


  [2/4] Apportioning census blocks to precincts...


INFO:   Running gpd.overlay intersection (may take 1-2 minutes)...
INFO:   Apportionment complete. Total apportioned population: 6,154,905
INFO:   Found 9 presidential candidate columns for 2024.


  [3/4] Extracting vote totals and calculating turnout...
  [4/4] Downscaling ACS demographics to precinct level...
  Done. 3,076 precincts assembled for 2024.

=== Precinct feature summary ===
  2016: 3,323 precincts | 21 columns
    Median turnout: 61.8%
    Columns: ['composite_prec_id', 'geometry', 'rep_votes', 'dem_votes', 'other_votes', 'total_votes', 'apportioned_population', 'apportioned_vap', 'turnout_pct', 'rep_pct', 'dem_pct', 'COUNTYFP', 'median_household_income', 'total_pop_25_plus', 'pct_bachelors_plus', 'pct_white', 'pct_minority', 'total_workers', 'pct_no_vehicle', 'pct_voting_age', 'year']
  2020: 3,733 precincts | 21 columns
    Median turnout: 66.6%
    Columns: ['composite_prec_id', 'geometry', 'rep_votes', 'dem_votes', 'other_votes', 'total_votes', 'apportioned_population', 'apportioned_vap', 'turnout_pct', 'rep_pct', 'dem_pct', 'COUNTYFP', 'median_household_income', 'total_pop_25_plus', 'pct_bachelors_plus', 'pct_white', 'pct_minority', 'total_workers', 'pct_no_ve

## Stage 7 — County Aggregation

Rolls precinct-level data to county level:
vote counts and population are summed; rates are population-weighted averaged;
precinct geometries are unioned into county polygons.

In [8]:
# Detect the county column name carried through by build_precinct_features
county_col = next(
    c for c in COUNTY_FIPS_CANDIDATES
    if c in precinct_features[ELECTION_YEARS[0]].columns
)
print(f"County column detected: '{county_col}'")

county_features = {}

for year in ELECTION_YEARS:
    county_features[year] = aggregate_to_county(
        precinct_features[year],
        precinct_id_col = PRECINCT_ID_COL,
        county_col      = county_col,
    )
    print(f'  {year}: {len(county_features[year])} counties')

print('\nCounty aggregation complete.')

INFO: Aggregating precinct data to county level...


County column detected: 'COUNTYFP'


INFO: County aggregation complete: 115 counties.
INFO: Aggregating precinct data to county level...


  2016: 115 counties


INFO: County aggregation complete: 115 counties.
INFO: Aggregating precinct data to county level...


  2020: 115 counties


INFO: County aggregation complete: 115 counties.


  2024: 115 counties

County aggregation complete.


## Stage 8 — Save Outputs

Writes tabular CSVs (for modeling) and GeoPackages (for GIS visualization).

In [9]:
import warnings

# Tabular — all years stacked
all_precinct = pd.concat(
    [precinct_features[y].drop(columns='geometry', errors='ignore') for y in ELECTION_YEARS],
    ignore_index=True,
)
all_county = pd.concat(
    [county_features[y].drop(columns='geometry', errors='ignore') for y in ELECTION_YEARS],
    ignore_index=True,
)

precinct_csv = PROCESSED / 'precinct_features_all_years.csv'
county_csv   = PROCESSED / 'county_features_all_years.csv'

all_precinct.to_csv(precinct_csv, index=False)
all_county.to_csv(county_csv,     index=False)

print(f'Saved: {precinct_csv}  ({len(all_precinct):,} rows)')
print(f'Saved: {county_csv}  ({len(all_county):,} rows)')

# Geospatial — one GeoPackage per year
for year in ELECTION_YEARS:
    prec_gpkg = GEO_OUT / f'precinct_features_{year}.gpkg'
    cnty_gpkg = GEO_OUT / f'county_features_{year}.gpkg'
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        precinct_features[year].to_file(prec_gpkg, driver='GPKG')
        county_features[year].to_file(cnty_gpkg,   driver='GPKG')
    print(f'Saved: {prec_gpkg}')
    print(f'Saved: {cnty_gpkg}')

print('\nAll outputs saved successfully.')

Saved: C:\Users\daves\OneDrive\Documents\School\SP26\AnalyticsApps\final\AnalyticsApps\data\processed\precinct_features_all_years.csv  (10,132 rows)
Saved: C:\Users\daves\OneDrive\Documents\School\SP26\AnalyticsApps\final\AnalyticsApps\data\processed\county_features_all_years.csv  (345 rows)


INFO: Created 3,323 records
INFO: Created 115 records


Saved: C:\Users\daves\OneDrive\Documents\School\SP26\AnalyticsApps\final\AnalyticsApps\data\geo\output\precinct_features_2016.gpkg
Saved: C:\Users\daves\OneDrive\Documents\School\SP26\AnalyticsApps\final\AnalyticsApps\data\geo\output\county_features_2016.gpkg


INFO: Created 3,733 records
INFO: Created 115 records


Saved: C:\Users\daves\OneDrive\Documents\School\SP26\AnalyticsApps\final\AnalyticsApps\data\geo\output\precinct_features_2020.gpkg
Saved: C:\Users\daves\OneDrive\Documents\School\SP26\AnalyticsApps\final\AnalyticsApps\data\geo\output\county_features_2020.gpkg


INFO: Created 3,076 records
INFO: Created 115 records


Saved: C:\Users\daves\OneDrive\Documents\School\SP26\AnalyticsApps\final\AnalyticsApps\data\geo\output\precinct_features_2024.gpkg
Saved: C:\Users\daves\OneDrive\Documents\School\SP26\AnalyticsApps\final\AnalyticsApps\data\geo\output\county_features_2024.gpkg

All outputs saved successfully.


## Diagnostics (optional)

In [10]:
# Turnout distribution by year
print('=== Precinct turnout by year ===')
for year in ELECTION_YEARS:
    gdf = precinct_features[year]
    if 'turnout_pct' in gdf.columns:
        t = gdf['turnout_pct'].dropna()
        print(f'  {year}: n={len(t):,}  median={t.median():.1f}%  '
              f'p10={t.quantile(0.10):.1f}%  p90={t.quantile(0.90):.1f}%  '
              f'>100pct: {(t > 100).sum()}')

=== Precinct turnout by year ===
  2016: n=3,282  median=61.8%  p10=35.1%  p90=77.2%  >100pct: 14
  2020: n=3,686  median=66.6%  p10=36.7%  p90=87.0%  >100pct: 55
  2024: n=3,076  median=67.0%  p10=38.5%  p90=88.6%  >100pct: 95


In [11]:
# Population coverage check
# Apportioned total should be close to Census block total (~6.1M for Missouri).
# Small differences (~1-2%) are expected for blocks straddling precinct boundaries.
print('=== Apportioned population vs Census block total ===')
block_total = blocks_gdf['total_population'].sum()
print(f'  Census blocks total: {block_total:,.0f}')
for year in ELECTION_YEARS:
    if 'apportioned_population' in precinct_features[year].columns:
        apportioned = precinct_features[year]['apportioned_population'].sum()
        coverage    = apportioned / block_total * 100
        print(f'  {year}: {apportioned:,.0f} apportioned  ({coverage:.1f}% coverage)')

=== Apportioned population vs Census block total ===
  Census blocks total: 6,154,913
  2016: 6,154,910 apportioned  (100.0% coverage)
  2020: 6,154,905 apportioned  (100.0% coverage)
  2024: 6,154,905 apportioned  (100.0% coverage)


## Map Review

Interactive Folium maps for visual QA of the pipeline outputs.
Each map reprojects to WGS84 (EPSG:4326) since Folium uses lat/lon.
Hover over a precinct to see its ID; click for the full attribute popup.

> **Note:** Run `pip install folium` if not already installed.

In [12]:
import folium

# Missouri center coordinates
MO_CENTER = [38.5, -92.5]
REVIEW_YEAR = 2020   # change to 2016 or 2024 to inspect other years

# Reproject to WGS84 for Folium (must be EPSG:4326)
gdf_review = precinct_features[REVIEW_YEAR].to_crs('EPSG:4326').copy()

# Folium needs a string key that matches a GeoJSON property
# composite_prec_id is already a string, so it works directly
geojson_data = gdf_review[[PRECINCT_ID_COL, 'geometry']].to_json()

print(f'Prepared {len(gdf_review):,} precincts for {REVIEW_YEAR} map review.')

Prepared 3,733 precincts for 2020 map review.


### Map 1 — Voter Turnout (%)

In [13]:
m_turnout = folium.Map(location=MO_CENTER, zoom_start=7, tiles='CartoDB positron')

folium.Choropleth(
    geo_data   = geojson_data,
    data       = gdf_review[[PRECINCT_ID_COL, 'turnout_pct']].dropna(),
    columns    = [PRECINCT_ID_COL, 'turnout_pct'],
    key_on     = f'feature.properties.{PRECINCT_ID_COL}',
    fill_color = 'YlOrRd',
    fill_opacity  = 0.7,
    line_opacity  = 0.15,
    line_weight   = 0.4,
    legend_name   = f'Voter Turnout (%) — {REVIEW_YEAR}',
    nan_fill_color = 'lightgray',
).add_to(m_turnout)

# Tooltip: hover to see precinct ID and turnout
folium.GeoJson(
    geojson_data,
    style_function  = lambda x: {'fillOpacity': 0, 'weight': 0},
    tooltip = folium.GeoJsonTooltip(
        fields  = [PRECINCT_ID_COL],
        aliases = ['Precinct:'],
        localize = True,
    ),
).add_to(m_turnout)

m_turnout.save('../data/geo/output/map_turnout_2020.html')


### Map 2 — Republican Vote Share (%)

In [14]:
m_party = folium.Map(location=MO_CENTER, zoom_start=7, tiles='CartoDB positron')

folium.Choropleth(
    geo_data   = geojson_data,
    data       = gdf_review[[PRECINCT_ID_COL, 'rep_pct']].dropna(),
    columns    = [PRECINCT_ID_COL, 'rep_pct'],
    key_on     = f'feature.properties.{PRECINCT_ID_COL}',
    fill_color = 'RdBu_r',   # Red = Republican-leaning, Blue = Democrat-leaning
    fill_opacity  = 0.7,
    line_opacity  = 0.15,
    line_weight   = 0.4,
    legend_name   = f'Republican Vote Share (%) — {REVIEW_YEAR}',
    nan_fill_color = 'lightgray',
).add_to(m_party)

# Tooltip: hover to see precinct ID
folium.GeoJson(
    geojson_data,
    style_function  = lambda x: {'fillOpacity': 0, 'weight': 0},
    tooltip = folium.GeoJsonTooltip(
        fields  = [PRECINCT_ID_COL],
        aliases = ['Precinct:'],
        localize = True,
    ),
).add_to(m_party)

m_party.save('../data/geo/output/map_party_2020.html')


### Map 3 — Apportioned Population

In [15]:
m_pop = folium.Map(location=MO_CENTER, zoom_start=7, tiles='CartoDB positron')

folium.Choropleth(
    geo_data   = geojson_data,
    data       = gdf_review[[PRECINCT_ID_COL, 'apportioned_population']].dropna(),
    columns    = [PRECINCT_ID_COL, 'apportioned_population'],
    key_on     = f'feature.properties.{PRECINCT_ID_COL}',
    fill_color = 'Blues',
    fill_opacity  = 0.7,
    line_opacity  = 0.15,
    line_weight   = 0.4,
    legend_name   = f'Apportioned Population — {REVIEW_YEAR}',
    nan_fill_color = 'lightgray',
).add_to(m_pop)

# Tooltip: hover to see precinct ID
folium.GeoJson(
    geojson_data,
    style_function  = lambda x: {'fillOpacity': 0, 'weight': 0},
    tooltip = folium.GeoJsonTooltip(
        fields  = [PRECINCT_ID_COL],
        aliases = ['Precinct:'],
        localize = True,
    ),
).add_to(m_pop)

# Use this to quickly spot precincts where apportionment looks implausible
# (very high or very low population relative to neighbors)
m_pop.save('../data/geo/output/map_pop_2020.html')
